# FedOPF: CFL-GP approach based on APPFL

* Need of making central unit class? => Single class that can manage all of server models, clustering period, ... etc.

In [1]:
import argparse
from omegaconf import OmegaConf
from appfl.agent import ClientAgent, ServerAgent

[W1223 02:07:52.902433725 Context.cpp:281] Warning: torch.backends.cuda.preferred_linalg_library is an experimental feature. If you see any error or unexpected behavior when this flag is set please file an issue on GitHub. (function operator())


In [2]:
cu_config_path = "./resources/configs/fedopf_central_unit_cfl_gp.yaml"
server_config_path = "./resources/configs/fedopf_server_cfl_gp.yaml"
client_config_path = "./resources/configs/fedopf_client_1.yaml"

client_ids = [14, 57, 60, 73, 89, 118, 162, 197, 250, 793, 1354, 1664]
num_clients = len(client_ids)

In [3]:
central_unit_config = OmegaConf.load(cu_config_path) # Load central unit configs for CFL-GP

# Load server agent config and set corresponding fields for # of clusters (CFL_GP)
n_models = central_unit_config.cu_configs.n_models
server_agent_configs = [
    OmegaConf.load(server_config_path) for _ in range(n_models)
]
# server_agent_config.server_configs.num_clients = num_clients

# Create server agent (CFL_GP)
server_agents = [ServerAgent(server_agent_config=server_agent_configs[i]) for i in range(n_models)]


appfl: ✅[2025-12-23 02:07:53,269 server]: Logging to ./output/result_Server_2025-12-23-02-07-53.txt


In [4]:
# Load base client configurations and set corresponding fields for different clients
client_agent_configs = [
    OmegaConf.load(client_config_path) for _ in range(num_clients)
]

for i, id in enumerate(client_ids):
    client_agent_configs[i].client_id = f"Client{i+1}"
    # client_agent_configs[i].data_configs.dataset_kwargs.num_clients = num_clients
    client_agent_configs[i].data_configs.dataset_kwargs.client_id = id
    # client_agent_configs[i].data_configs.dataset_kwargs.visualization = (
    #     True if i == 0 else False
    # )
    
    # only enable wandb for the first client is sufficient for logging all clients in serial run
    if hasattr(client_agent_configs[i], "wandb_configs") and client_agent_configs[i].wandb_configs.get("enable_wandb", False):
        if i == 0:
            client_agent_configs[i].wandb_configs.enable_wandb = True
        else:
            client_agent_configs[i].wandb_configs.enable_wandb = False

In [5]:
# Load client agents
client_agents = [
    ClientAgent(client_agent_config=client_agent_configs[i]) for i in range(num_clients)
]

appfl: ✅[2025-12-23 02:07:53,422 Client1]: Logging to ./output/result_Client1_2025-12-23-02-07-53.txt
/home/super/data1/skj/FedOPF-APPFL/CFL-GP/resources/dataset/acopf_prob.py:71: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /opt/conda/conda-bld/pytorch_1728929558238/work/torch/csrc/utils/tensor_new.cpp:278.)
  self.slackva = torch.tensor([np.deg2rad(ppc['bus'][self.slack, idx_bus.VA])],
appfl: ✅[2025-12-23 02:07:53,765 Client2]: Logging to ./output/result_Client2_2025-12-23-02-07-53.txt
appfl: ✅[2025-12-23 02:07:54,209 Client3]: Logging to ./output/result_Client3_2025-12-23-02-07-54.txt
appfl: ✅[2025-12-23 02:07:54,842 Client4]: Logging to ./output/result_Client4_2025-12-23-02-07-54.txt
appfl: ✅[2025-12-23 02:07:55,332 Client5]: Logging to ./output/result_Client5_2025-12-23-02-07-55.txt
appfl: ✅[2025-12-23 02:07:55,

In [6]:
# Get additional client configurations from the server
client_config_from_server = server_agents[0].get_client_configs()
# print(OmegaConf.to_yaml(server_agent.get_client_configs()))
client_config_from_server.model_configs.model_name = 'GraphOPF'
client_config_from_server.model_configs.model_path = "./resources/model/graphopf_client.py"

for client_agent in client_agents:
    client_config_from_server.model_configs.model_kwargs.client_id = client_agent.dataset.nbus
    client_agent.load_config(client_config_from_server)

In [7]:
central_unit_config.cu_configs

{'n_models': 4, 'clustering_period': 2, 'clustering_termination_threshold': 50, 'cu_compressor_configs': {'use_reduced_G': False, 'gradient_compression_ratio': 100}, 'num_comm_rounds': 200, 'unknown_k': False, 'warmup_epoch': 0}

In [8]:
import torch
import numpy as np 
from utils.cfl_gp import get_num_cluster, spectral_clustering_and_matching
from collections import OrderedDict # NOTE: kj
import time
import copy

# Main code: run CFL-GP
# CFL-GP parameters
clustering_period = central_unit_config.cu_configs.clustering_period
use_reduced_G = central_unit_config.cu_configs.cu_compressor_configs.use_reduced_G
gradient_compression_ratio = central_unit_config.cu_configs.cu_compressor_configs.gradient_compression_ratio
clustering_termination_threshold = central_unit_config.cu_configs.clustering_termination_threshold
estimated_cluster_ids = None
estimated_cluster_ids_new = None
unknown_k = central_unit_config.cu_configs.unknown_k
warmup_epoch = central_unit_config.cu_configs.warmup_epoch

n_params_per_model = sum(param.numel() for param in server_agents[0].model.parameters())
n_params_compressed_gradient = n_params_per_model
if use_reduced_G:
    n_params_compressed_gradient = n_params_per_model // gradient_compression_ratio
    random_G_indices = np.random.choice(np.arange(n_params_per_model), size=n_params_compressed_gradient, replace=False)
print("n_params_per_model: ", n_params_per_model)
print("n_params_compressed_gradient: ", n_params_compressed_gradient)

gradient_profile_matrix = np.zeros(shape=(n_models * n_params_compressed_gradient, num_clients))
criterion_model_index = 0
CLUSTERING_CONVERGENCE = False
metrics = {}
for round in range(central_unit_config.cu_configs.num_comm_rounds):
    # Model save
    if round == central_unit_config.cu_configs.num_comm_rounds-1:
        for client_agent in client_agents:
            client_agent.save_checkpoint()
        for m in range(n_models):
            server_agents[m].save_checkpoint(server_id=m)

    if round != 0:
        # Load the new global model from the server
        for cluster_idx, client_cluster in enumerate(client_clusters):
            print("load new group model")
            for client, new_global_model_future in zip(client_cluster, group_global_models[cluster_idx]):
                client.load_parameters(new_global_model_future.result())

    cr_start_time = time.time()
    for c_idx, client_agent in enumerate(client_agents):
        # - Select models for downlink transmission
        if round % clustering_period == 0 and CLUSTERING_CONVERGENCE is not True:
            if round == 0:
                query_model_indicies = [0]
            elif estimated_cluster_ids[c_idx] == criterion_model_index:
                query_model_indicies = [estimated_cluster_ids[c_idx]]
            else:
                query_model_indicies = [criterion_model_index, estimated_cluster_ids[c_idx]]
        else:
            query_model_indicies = [estimated_cluster_ids[c_idx]]
        
        # - local update or gradient calculation
        for model_idx in query_model_indicies:
            if round == 0:
                estimated_cluster_id = 0
                # Load initial global model from the server
                init_global_model = server_agents[estimated_cluster_id].get_parameters(serial_run=True)
                client_agent.load_parameters(init_global_model)
            else:
                estimated_cluster_id = estimated_cluster_ids[c_idx]  
                print(server_agents[estimated_cluster_id].get_parameters(init_model=False)['layers.0.edge_aggr.0.weight'][:2,:])
        

            ## client local training
            client_agent.train(round=round)
            local_model = client_agent.get_parameters()
            if isinstance(local_model, tuple):
                local_model, metadata = local_model[0], local_model[1]
            else:
                metadata = {}
            ## NOTE: kj
            local_model = OrderedDict((k,v) for k,v in local_model.items() if k.startswith("layers"))
            
            if round % clustering_period == 0 and model_idx == criterion_model_index and CLUSTERING_CONVERGENCE is not True:
                # vectorized_model_info = flatten_tensor(local_model).clone().cpu().detach().numpy() ~~~ # <== shared NNs 이 들어가야 함! "local_model" 자체가 애초에 shared layers를 불러오게끔 세팅되어 있음.
                vectorized_model_info = torch.cat([value.flatten() for value in local_model.values()])
                vectorized_model_info = vectorized_model_info.clone().cpu().detach().numpy()
                if use_reduced_G is True:
                    selected_vectorized_model_info = vectorized_model_info[random_G_indices]
                    vectorized_model_info = selected_vectorized_model_info

                # Cumulative Averaging
                beta = (1 / (np.floor((round + 1) / (n_models * clustering_period)) + 1))
                # global_logger.info("beta:{}".format(beta))
                gradient_profile_matrix[(criterion_model_index)*n_params_compressed_gradient : (criterion_model_index + 1)*n_params_compressed_gradient,c_idx] = \
                    gradient_profile_matrix[(criterion_model_index)*n_params_compressed_gradient : (criterion_model_index + 1)*n_params_compressed_gradient,c_idx] * \
                        (1 - beta) + (beta) * vectorized_model_info

    # - Clustering & Matching
    if unknown_k is True and round < warmup_epoch:
        proposed_k = get_num_cluster(gradient_profile_matrix, n_centers=n_models,
                                        n_clients=num_clients,
                                        estimated_cluster_ids_old=estimated_cluster_ids)
        estimated_cluster_ids = np.zeros(shape=num_clients, dtype=int)
        estimated_cluster_ids_new = np.zeros(shape=num_clients, dtype=int)
        consistency_cnt = 0
    elif unknown_k is True and round == warmup_epoch:
        n_models = get_num_cluster(gradient_profile_matrix, n_centers=n_models,
                                        n_clients=num_clients,
                                        estimated_cluster_ids_old=estimated_cluster_ids)
        group_global_models = [[] for _ in range(n_models)]
        print("Set number of clusters as {}.".format(n_models))

        for m in np.arange(1, n_models):
            # copy_weight2(target=self.models[m], source=self.models[0])
            server_agents[m].model.load_state_dict(server_agents[0].get_parameters(serial_run=True))
    
        estimated_cluster_ids = np.zeros(shape=num_clients, dtype=int)
        estimated_cluster_ids_new = np.zeros(shape=num_clients, dtype=int)
        consistency_cnt = 0
    elif round % clustering_period == 0 and round < clustering_termination_threshold:
        print("sepctral_clustering_and_matching")
        # print(gradient_profile_matrix.shape)
        criterion_model_index = (criterion_model_index + 1) % n_models
        info = spectral_clustering_and_matching(gradient_profile_matrix, n_centers=n_models,
                                                n_clients=num_clients,
                                                estimated_cluster_ids_old=estimated_cluster_ids,
                                                clustering_algorithm='KMeans') # KMeans, DBSCAN, AgglomerativeClustering
        estimated_cluster_ids_new = info["estimated_cluster_ids"]  # update cluster ids.
        reduced_gradient_profile_matrix = info["reduced_gradient_profile_matrix"]
        singular_values = info["singular_values"]

        estimated_cluster_ids = estimated_cluster_ids_new  # np.zeros(shape=self.n_clients, dtype=int)        

    print("cluster ids: {}".format(estimated_cluster_ids))

    # - Model update
    group_global_models = [[] for _ in range(n_models)]
    clustered_client_indices = [np.where(estimated_cluster_ids == cluster_id)[0] for cluster_id in range(n_models)]
    client_clusters = [[client_agents[i] for i in client_indices] for client_indices in clustered_client_indices]
    ## 1. 
    for cluster_idx, client_cluster in enumerate(client_clusters):
        if len(client_cluster) >= 1:
            # print("group model update!")
            server_agents[cluster_idx].num_clients = len(client_cluster)
            # server_agents[cluster_idx]._set_num_clients()

            # TODO: Check handling server configs for clustered FL. 
            server_agents[cluster_idx].aggregator.aggregator_configs.num_clients = len(client_cluster)
            server_agents[cluster_idx].scheduler.num_clients = len(client_cluster)
            # server_agents[cluster_idx]._load_scheduler()
            # server_agents[cluster_idx].scheduler.aggregation_kwargs['num_clients'] = len(client_cluster)

            for client in client_cluster:
                c_model = client.get_parameters()
                if isinstance(c_model, tuple):
                    c_model, metadata = c_model[0], c_model[1]
                else:
                    metadata = {}

                ## NOTE: kj
                c_model = OrderedDict((k,v) for k,v in c_model.items() if k.startswith("layers"))

                # "Send" local model to server and get a Future object for the new global model
                # The Future object will be resolved when the server receives local models from all clients
                new_global_model_future = server_agents[cluster_idx].global_update(
                    client_id=client.get_id(),
                    local_model=c_model,
                    blocking=False, # False
                    **metadata,
                )
                group_global_models[cluster_idx].append(new_global_model_future)

    time_for_communication_round = time.time() - cr_start_time
    # ====================================
    # Data Tracking and save Results
    # ====================================
    data_tracking = {
        "estimated_cluster_ids": copy.deepcopy(estimated_cluster_ids),
        "reduced_gradient_profile_matrix": copy.deepcopy(reduced_gradient_profile_matrix),
        # "singular_values": copy.deepcopy(self.singular_values),
        "time_for_communication_round": time_for_communication_round
    }
    metrics["c_round_" + str(round)] = data_tracking
    if round == central_unit_config.cu_configs.num_comm_rounds-1: 
        server_agents[0].save_data_tracking(metrics) # just use save function, so choose random server agent 

n_params_per_model:  462320
n_params_compressed_gradient:  462320


appfl: ✅[2025-12-23 02:08:16,585 Client1]:      Round      Epoch       Time Train Loss Train Accuracy
appfl: ✅[2025-12-23 02:08:17,319 Client1]:          0          0     0.7274     1.4128           40.0
appfl: ✅[2025-12-23 02:08:17,441 Client1]:          0          1     0.1196     1.3679           40.0
appfl: ✅[2025-12-23 02:08:17,563 Client1]:          0          2     0.1209     1.3223           40.0
appfl: ✅[2025-12-23 02:08:17,696 Client1]:          0          3     0.1298     1.2721           40.0
appfl: ✅[2025-12-23 02:08:17,812 Client1]:          0          4     0.1145     1.2111           40.0
appfl: ✅[2025-12-23 02:08:21,006 Client2]:      Round      Epoch       Time Train Loss Train Accuracy
appfl: ✅[2025-12-23 02:08:21,149 Client2]:          0          0     0.1369     6.7652       44.57143
appfl: ✅[2025-12-23 02:08:21,278 Client2]:          0          1     0.1269     6.5817       46.57143
appfl: ✅[2025-12-23 02:08:21,399 Client2]:          0          2     0.1186     6.

sepctral_clustering_and_matching
cluster ids: [2 3 0 3 0 3 0 0 3 1 1 3]
load new group model
load new group model
load new group model
load new group model


appfl: ✅[2025-12-23 02:09:59,863 Client1]:          1          0     0.0823     1.1448           40.0
appfl: ✅[2025-12-23 02:09:59,957 Client1]:          1          1     0.0920     1.0620           40.0


tensor([[ 0.2801,  0.3017, -0.0797,  0.3342, -0.0742,  0.0702, -0.1637,  0.2047],
        [ 0.3178, -0.2509,  0.3166,  0.0718,  0.2712,  0.0579,  0.1702, -0.0598]])


...

tensor([[ 0.2549,  0.2855, -0.0934,  0.3343, -0.0561,  0.0908, -0.1536,  0.1718],
        [ 0.3330, -0.2537,  0.2752,  0.0307,  0.1850, -0.0236,  0.2239,  0.0090]])
warm up end!


appfl: ✅[2025-12-23 09:22:47,592 Client11]:        199          0     2.9902   137.2291       88.73078
appfl: ✅[2025-12-23 09:22:50,574 Client11]:        199          1     2.9807   137.1884       90.00768
appfl: ✅[2025-12-23 09:22:53,567 Client11]:        199          2     2.9911   136.8445       92.41538
appfl: ✅[2025-12-23 09:22:56,551 Client11]:        199          3     2.9832   135.0229       92.63077
appfl: ✅[2025-12-23 09:22:59,531 Client11]:        199          4     2.9784   135.0653       90.46153


tensor([[ 0.3012,  0.2287, -0.1000,  0.3896,  0.0235,  0.1920, -0.2617,  0.0996],
        [ 0.4334, -0.2865,  0.3890, -0.0258,  0.2149,  0.0691,  0.2593,  0.0788]])
warm up end!


appfl: ✅[2025-12-23 09:23:05,843 Client12]:        199          0     4.5348    22.3973       98.89742
appfl: ✅[2025-12-23 09:23:10,225 Client12]:        199          1     4.3803    22.3696        99.5641
appfl: ✅[2025-12-23 09:23:14,620 Client12]:        199          2     4.3938    22.3538      99.512825
appfl: ✅[2025-12-23 09:23:19,091 Client12]:        199          3     4.4698    22.3657      99.128204
appfl: ✅[2025-12-23 09:23:23,529 Client12]:        199          4     4.4370    22.3598        99.4359


cluster ids: [2 3 0 3 0 0 0 0 3 1 1 0]


In [9]:
estimated_cluster_ids

array([2, 3, 0, 3, 0, 0, 0, 0, 3, 1, 1, 0])

## Evaluate model on "test dataset" for each client

In [10]:
# Modifies stats in place
def dict_agg(stats, key, value, op='concat'):
    if key in stats.keys():
        if op == 'sum':
            stats[key] += value
        elif op == 'concat':
            stats[key] = np.concatenate((stats[key], value), axis=0)
        else:
            raise NotImplementedError
    else:
        stats[key] = value

In [ ]:
from torch_geometric.loader import DataLoader
from pypower.api import makeYbus
import torch
import numpy as np
import time

for client_agent in client_agents:
    print(client_agent.get_id())
    test_dataloader = DataLoader(
            client_agent.test_dataset,
            batch_size=1,
            shuffle=False,
            drop_last=True
        )

    Ybus, Yf, Yt = makeYbus(client_agent.dataset.baseMVA, client_agent.dataset.ppc['bus'], client_agent.dataset.ppc['branch'])
    # branch thermal limit information
    flow_max = (client_agent.dataset.ppc['branch'][:, 5] / client_agent.dataset.baseMVA)**2
    flow_max[flow_max == 0] = np.inf # np.Inf
    flow_max = torch.tensor(flow_max, dtype=torch.float32).to(client_agent.dataset.device)

    test_len = 200
    node_means, node_stds, edge_means, edge_stds = client_agent.dataset.input_standardization(test_len, train=False) # (1, 2*nbus) <= for data normalization
    n_means = node_means.to(client_agent.dataset.device)
    n_stds = node_stds.to(client_agent.dataset.device)
    e_means = edge_means.to(client_agent.dataset.device)
    e_stds = edge_stds.to(client_agent.dataset.device)

    client_agent.model.eval()
    test_stats = {}
    test_eps_converge = 1e-4

    # LagM = torch.ones(1, 2*ng + 2*nbus + 2*nl).to(DEVICE) # shape: (1, num_inequalities)
    LagM_sp_g = torch.ones(1, 2).to(client_agent.dataset.device) # shape: (1, num_inequalities)
    LagM_q_g = torch.ones(1, 2*client_agent.dataset.ng).to(client_agent.dataset.device) # shape: (1, num_inequalities)
    LagM_v_m = torch.ones(1, 2*client_agent.dataset.nbus).to(client_agent.dataset.device) # shape: (1, num_inequalities)
    LagM_line_l = torch.ones(1, 2*client_agent.dataset.nl).to(client_agent.dataset.device) # shape: (1, num_inequalities)

    solve_time = []
    with torch.no_grad():
        for (i, Xtest) in enumerate(test_dataloader):
            Xtest = Xtest.to(client_agent.dataset.device)

            start_time = time.time()
            Y = client_agent.model(Xtest, n_means, n_stds, e_means, e_stds)
            end_time = time.time()

            solve_time += [end_time - start_time]

            ## line thermal limit
            pg, qg, vm, va = client_agent.dataset.get_yvars(Y)
            vr = vm*torch.cos(va)
            vi = vm*torch.sin(va)
            vz = torch.complex(vr, vi) # complex voltage

            # calculate the branch current of from bus and to bus based on the Yf*V and Yt*V
            If = torch.tensor(Yf.todense(), dtype=torch.complex64).to(client_agent.dataset.device) @ vz.T
            It = torch.tensor(Yt.todense(), dtype=torch.complex64).to(client_agent.dataset.device) @ vz.T

            # Calculate the apparent power S
            Sf = vz[:,client_agent.dataset.ppc['branch'][:,0].astype(int)] * torch.conj(If.T)
            St = vz[:,client_agent.dataset.ppc['branch'][:,1].astype(int)] * torch.conj(It.T)
            Sff = Sf * torch.conj(Sf)
            Stt = St * torch.conj(St)

            # calculate the line thermal limit constraints violation
            diff_Sf = Sff.real - flow_max
            diff_St = Stt.real - flow_max
            # diff_Sf[torch.clamp(diff_Sf, 0) != 0]

            line_limit_vio_Sf = torch.clamp(diff_Sf, 0)
            line_limit_vio_St = torch.clamp(diff_St, 0)
            ###########################################

            # test_loss, test_obj_cost, test_ineq_dist, test_eq_resid = total_loss(data, Xtest.x, Y, LagM)
            test_loss, test_obj_cost, test_ineq_dist, test_eq_resid = client_agent.loss_fn.forward(client_agent.dataset, Xtest.x, Y, LagM_sp_g, LagM_q_g, LagM_v_m, LagM_line_l) # LagM is lagrangian multiplier, and the shape is (1, num_inequalities)

            dict_agg(test_stats, 'time', np.array(end_time - start_time).reshape(1,-1))

            test_ineq_p_g = torch.cat([pg - client_agent.dataset.pmax.to(client_agent.dataset.device), client_agent.dataset.pmin.to(client_agent.dataset.device) - pg], dim=1)
            test_ineq_p_g = torch.clamp(test_ineq_p_g, 0).to(client_agent.dataset.device)
            test_ineq_q_g = test_ineq_dist[:,2:2+2*client_agent.dataset.ng]
            test_ineq_v_m = test_ineq_dist[:,2+2*client_agent.dataset.ng:2+2*client_agent.dataset.ng+2*client_agent.dataset.nbus]
            test_ineq_line_l = test_ineq_dist[:,2+2*client_agent.dataset.ng+2*client_agent.dataset.nbus:]

            dict_agg(test_stats, 'test_loss', test_loss.detach().cpu().numpy())
            # dict_agg(test_stats, 'test_loss', (test_loss[0]+test_loss[1]+test_loss[2]+test_loss[3]).detach().cpu().numpy())

            dict_agg(test_stats, 'test_obj_cost', test_obj_cost.detach().cpu().numpy())

            dict_agg(test_stats, 'test_ineq_max', torch.max(test_ineq_dist, dim=1)[0].detach().cpu().numpy())
            dict_agg(test_stats, 'test_ineq_mean', torch.mean(test_ineq_dist, dim=1).detach().cpu().numpy())
            dict_agg(test_stats, 'test_ineq_p_g_max', torch.max(test_ineq_p_g, dim=1)[0].detach().cpu().numpy())
            dict_agg(test_stats, 'test_ineq_p_g_mean', torch.mean(test_ineq_p_g, dim=1).detach().cpu().numpy())
            dict_agg(test_stats, 'test_ineq_q_g_max', torch.max(test_ineq_q_g, dim=1)[0].detach().cpu().numpy())
            dict_agg(test_stats, 'test_ineq_q_g_mean', torch.mean(test_ineq_q_g, dim=1).detach().cpu().numpy())
            dict_agg(test_stats, 'test_ineq_v_m_max', torch.max(test_ineq_v_m, dim=1)[0].detach().cpu().numpy())
            dict_agg(test_stats, 'test_ineq_v_m_mean', torch.mean(test_ineq_v_m, dim=1).detach().cpu().numpy())
            dict_agg(test_stats, 'test_ineq_line_l_max', torch.max(test_ineq_line_l, dim=1)[0].detach().cpu().numpy())
            dict_agg(test_stats, 'test_ineq_line_l_mean', torch.mean(test_ineq_line_l, dim=1).detach().cpu().numpy())

            pg_rate_torch = (((pg <= client_agent.dataset.pmax.to(client_agent.dataset.device)) & (pg >= client_agent.dataset.pmin.to(client_agent.dataset.device))).sum()/client_agent.dataset.ng)*100
            qg_rate_torch = (((qg <= client_agent.dataset.qmax.to(client_agent.dataset.device)) & (qg >= client_agent.dataset.qmin.to(client_agent.dataset.device))).sum()/client_agent.dataset.ng)*100
            dict_agg(test_stats, 'test_p_g_satisfication rate (%)', pg_rate_torch.detach().cpu().numpy().reshape(-1,1))
            dict_agg(test_stats, 'test_q_g_satisfication rate (%)', qg_rate_torch.detach().cpu().numpy().reshape(-1,1))
            # dict_agg(test_stats, 'test_p_g_satisfication rate (%)', ((torch.sum(test_ineq_p_g == 0, dim=1)/test_ineq_p_g.shape[1])*100).detach().cpu().numpy())
            # dict_agg(test_stats, 'test_q_g_satisfication rate (%)', ((torch.sum(test_ineq_q_g == 0, dim=1)/test_ineq_q_g.shape[1])*100).detach().cpu().numpy())

            v_rate_torch = (((vm <= client_agent.dataset.vmax.to(client_agent.dataset.device)) & (vm >= client_agent.dataset.vmin.to(client_agent.dataset.device))).sum()/client_agent.dataset.nbus)*100
            dict_agg(test_stats, 'test_v_m_satisfication rate (%)', v_rate_torch.detach().cpu().numpy().reshape(-1,1))
            # dict_agg(test_stats, 'test_v_m_satisfication rate (%)', ((torch.sum(test_ineq_v_m == 0, dim=1)/test_ineq_v_m.shape[1])*100).detach().cpu().numpy())

            sff_rate_torch = ((Sff.real <= flow_max).sum()/client_agent.dataset.nl)*100        
            stt_rate_torch = ((Stt.real <= flow_max).sum()/client_agent.dataset.nl)*100        
            dict_agg(test_stats, 'test_line_limit_satisfication_rate_Sf(%)', sff_rate_torch.detach().cpu().numpy().reshape(-1,1))
            dict_agg(test_stats, 'test_line_limit_satisfication_rate_St(%)', stt_rate_torch.detach().cpu().numpy().reshape(-1,1))
            dict_agg(test_stats, 'test_line_limit_satisfication_rate (%)', ((torch.sum(test_ineq_line_l == 0, dim=1)/test_ineq_line_l.shape[1])*100).detach().cpu().numpy())
            # dict_agg(test_stats, 'test_line_limit_satisfication_rate_Sf(%)', ((torch.sum(line_limit_vio_Sf == 0, dim=1)/line_limit_vio_Sf.shape[1])*100).detach().cpu().numpy())
            # dict_agg(test_stats, 'test_line_limit_satisfication_rate_St(%)', ((torch.sum(line_limit_vio_St == 0, dim=1)/line_limit_vio_St.shape[1])*100).detach().cpu().numpy())

            dict_agg(test_stats, 'test_ineq_q_g_num_viol_0', torch.sum(test_ineq_q_g > test_eps_converge, dim=1).detach().cpu().numpy())
            dict_agg(test_stats, 'test_ineq_v_m_num_viol_0', torch.sum(test_ineq_v_m > test_eps_converge, dim=1).detach().cpu().numpy())

            test_eq_real = test_eq_resid[:,:client_agent.dataset.nbus]
            test_eq_react = test_eq_resid[:,client_agent.dataset.nbus:]
            dict_agg(test_stats, 'test_eq_max', torch.max(torch.abs(test_eq_resid), dim=1)[0].detach().cpu().numpy())
            dict_agg(test_stats, 'test_eq_mean', torch.mean(torch.abs(test_eq_resid), dim=1).detach().cpu().numpy())
            dict_agg(test_stats, 'test_eq_real_max', torch.max(torch.abs(test_eq_real), dim=1)[0].detach().cpu().numpy())
            dict_agg(test_stats, 'test_eq_real_mean', torch.mean(torch.abs(test_eq_real), dim=1).detach().cpu().numpy())
            dict_agg(test_stats, 'test_eq_react_max', torch.max(torch.abs(test_eq_react), dim=1)[0].detach().cpu().numpy())
            dict_agg(test_stats, 'test_eq_react_mean', torch.mean(torch.abs(test_eq_react), dim=1).detach().cpu().numpy())
            dict_agg(test_stats, 'test_active_eq_satisfication rate (%)', (torch.sum((test_eq_resid[:,:client_agent.dataset.nbus] <= 1e-2) & (test_eq_resid[:,:client_agent.dataset.nbus] >= -1e-2)  , dim=1)/test_eq_resid[:,:client_agent.dataset.nbus].shape[1]*100).detach().cpu().numpy())
            dict_agg(test_stats, 'test_reactive_eq_satisfication rate (%)', (torch.sum((test_eq_resid[:,client_agent.dataset.nbus:] <= 1e-2) & (test_eq_resid[:,client_agent.dataset.nbus:] >= -1e-2)  , dim=1)/test_eq_resid[:,client_agent.dataset.nbus:].shape[1]*100).detach().cpu().numpy())

    print("GraphLDE obj. value for test samples: ", np.round(np.mean(test_stats['test_obj_cost'])*10000, 4))
    print("GraphLDE eq. mean for test samples: ", np.mean(test_stats['test_eq_mean']))
    print("GraphLDE eq. max for test samples: ", np.mean(test_stats['test_eq_max']))
    print("GraphLDE eq. active mean for test samples: ", np.mean(test_stats['test_eq_real_mean']))
    print("GraphLDE eq. active max for test samples: ", np.mean(test_stats['test_eq_real_max']))
    print("GraphLDE eq. reactive mean for test samples: ", np.mean(test_stats['test_eq_react_mean']))
    print("GraphLDE eq. reactive max for test samples: ", np.mean(test_stats['test_eq_react_max']))

    print("\n")
    print("GraphLDE ineq. mean for test samples: ", np.mean(test_stats['test_ineq_mean']))
    print("GraphLDE ineq. max for test samples: ", np.mean(test_stats['test_ineq_max']))
    print("GraphLDE ineq. p_g mean for test samples: ", np.mean(test_stats['test_ineq_p_g_mean']))
    print("GraphLDE ineq. p_g max for test samples: ", np.mean(test_stats['test_ineq_p_g_max']))
    print("GraphLDE ineq. q_g mean for test samples: ", np.mean(test_stats['test_ineq_q_g_mean']))
    print("GraphLDE ineq. q_g max for test samples: ", np.mean(test_stats['test_ineq_q_g_max']))
    print("GraphLDE ineq. v_m mean for test samples: ", np.mean(test_stats['test_ineq_v_m_mean']))
    print("GraphLDE ineq. v_m max for test samples: ", np.mean(test_stats['test_ineq_v_m_max']))
    print("GraphLDE ineq. line_l mean for test samples: ", np.mean(test_stats['test_ineq_line_l_mean']))
    print("GraphLDE ineq. line_l max for test samples: ", np.mean(test_stats['test_ineq_line_l_max']))

    print("\n")
    print("GraphLDE p_g satisfication rate for test samples: ", np.mean(test_stats['test_p_g_satisfication rate (%)']))
    print("GraphLDE q_g satisfication rate for test samples: ", np.mean(test_stats['test_q_g_satisfication rate (%)']))
    print("GraphLDE v_m satisfication rate for test samples: ", np.mean(test_stats['test_v_m_satisfication rate (%)']))
    print("GraphLDE test_line_limit_satisfication_rate_Sf for test samples: ", np.mean(test_stats['test_line_limit_satisfication_rate_Sf(%)']))
    print("GraphLDE test_line_limit_satisfication_rate_St for test samples: ", np.mean(test_stats['test_line_limit_satisfication_rate_St(%)']))
    print("GraphLDE active eq satisfication rate for test samples: ", np.mean(test_stats['test_active_eq_satisfication rate (%)']))
    print("GraphLDE reactive eq satisfication rate for test samples: ", np.mean(test_stats['test_reactive_eq_satisfication rate (%)']))

    print("\n")
    print("GraphLDE time (ms) <== average value for test dataset:", (np.mean(test_stats['time']*1e3)))
    print('\n')

# [14, 57, 60, 73, 89, 118, 162, 197, 250, 793, 1354, 1664]

Client1
GraphLDE obj. value for test samples:  2174.8719
GraphLDE eq. mean for test samples:  8.23152e-05
GraphLDE eq. max for test samples:  0.0006076633
GraphLDE eq. active mean for test samples:  7.59476e-05
GraphLDE eq. active max for test samples:  0.00023235858
GraphLDE eq. reactive mean for test samples:  8.8682806e-05
GraphLDE eq. reactive max for test samples:  0.0006076574


GraphLDE ineq. mean for test samples:  8.587626e-06
GraphLDE ineq. max for test samples:  0.0006870101
GraphLDE ineq. p_g mean for test samples:  0.0
GraphLDE ineq. p_g max for test samples:  0.0
GraphLDE ineq. q_g mean for test samples:  6.870101e-05
GraphLDE ineq. q_g max for test samples:  0.0006870101
GraphLDE ineq. v_m mean for test samples:  0.0
GraphLDE ineq. v_m max for test samples:  0.0
GraphLDE ineq. line_l mean for test samples:  0.0
GraphLDE ineq. line_l max for test samples:  0.0


GraphLDE p_g satisfication rate for test samples:  100.0
GraphLDE q_g satisfication rate for test samples:  98.